# phase_2 — ZERO-SHOT baseline (7B / 14B, NO finetuning)

**Standalone** notebook (independent of the finetune one). Measures how accurate larger instruct
models are with **no finetuning** — only the strict prompt (hard rules + an inline example, which
lists the 69 allowed findings). This is the bar the finetuned 3B must clear, and tells you whether
finetuning beats just prompting a bigger model.

Uses `eval_sg_llm.py --prompt strict` with **no `--lora`** (= base model), 4-bit so 7B/14B fit on T4.
No Drive needed — the per-finding metrics are printed here and the JSONs land in `/kaggle/working`
(Output tab). Just **Run All** on a GPU session (1×T4 is enough; 2×T4 fine too).

## CONFIG — edit these

In [ ]:
import os
PHASE2_SRC = "/kaggle/working/repo/phase_2"
SG_SFT  = "/kaggle/input/datasets/nguynnghin/sg-sft"   # build_sft_dataset.py output (val/test.jsonl)

ZS_MODELS = ["Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen2.5-14B-Instruct"]   # base models to probe
ZS_LIMIT  = 200      # number of samples (zero-shot ballpark)
ZS_SPLIT  = "val"    # "val" or "test"
ZS_BATCH  = 4        # generation batch (lower if 14B OOMs on a single T4)

for k, v in dict(PHASE2_SRC=PHASE2_SRC, SG_SFT=SG_SFT, ZS_LIMIT=ZS_LIMIT,
                 ZS_SPLIT=ZS_SPLIT, ZS_BATCH=ZS_BATCH).items():
    os.environ[k] = str(v)
print("config | models =", ZS_MODELS, "| limit =", ZS_LIMIT, "| split =", ZS_SPLIT)


In [ ]:
# --- code from GitHub. Internet ON. ---
!rm -rf /kaggle/working/repo && git clone -q https://github.com/hiennguyendang/phase_2_3_4_5 /kaggle/working/repo
!ls /kaggle/working/repo/phase_2 | head


## 1 — copy code + install deps (pinned)
`transformers` pinned to 4.45.2 (Kaggle's default v5 changed APIs). Copies `hedge.py` next to
`phase_2` so `sg_schema` imports it, and auto-resolves the dataset path.

In [ ]:
!rm -rf /kaggle/working/phase_2 && cp -r "$PHASE2_SRC" /kaggle/working/phase_2
!cp "$PHASE2_SRC/../hedge.py" /kaggle/working/hedge.py   # shared module: ancestor of phase_2 -> importable
%cd /kaggle/working/phase_2
!pip install -q "transformers==4.45.2" "accelerate>=0.34,<1.2" "bitsandbytes>=0.43"

# --- resolve SG_SFT robustly (dataset may mount under a different slug or a nested folder) ---
import os, glob, pathlib
sg = os.environ["SG_SFT"]
probe = "%s.jsonl" % os.environ["ZS_SPLIT"]
if not os.path.exists(os.path.join(sg, probe)):
    hits = glob.glob("/kaggle/input/**/%s" % probe, recursive=True)
    if hits:
        sg = str(pathlib.Path(hits[0]).parent); os.environ["SG_SFT"] = sg
        print("auto-resolved SG_SFT ->", sg)
    else:
        print("[ERROR] no %s under /kaggle/input -> ATTACH the sg-sft dataset (Add Input)." % probe)
        print(os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "no /kaggle/input")
print("SG_SFT =", os.environ["SG_SFT"])
if os.path.isdir(os.environ["SG_SFT"]):
    print("files:", os.listdir(os.environ["SG_SFT"]))


## 2 — run zero-shot (strict prompt, base models)
One `eval_sg_llm.py` per model. `--prompt strict` swaps in the tighter zero-shot prompt; no `--lora`
= the raw base model. fp16 is auto-picked on T4. ~200 samples × 2 models ≈ 10-20 min.

In [ ]:
import os
SG = os.environ["SG_SFT"]
for m in ZS_MODELS:
    tag = m.split("/")[-1]
    out = "/kaggle/working/sg_eval_zeroshot_%s.json" % tag
    print("\n========== ZERO-SHOT %s (strict prompt) ==========" % m)
    cmd = ('python eval_sg_llm.py --val "%s/%s.jsonl" --model "%s" --prompt strict '
           '--out "%s" --limit %s --batch %s') % (
           SG, os.environ["ZS_SPLIT"], m, out, os.environ["ZS_LIMIT"], os.environ["ZS_BATCH"])
    print(cmd)
    os.system(cmd)


## 3 — read the results
Each `sg_eval_zeroshot_<model>.json` (and the printout above) has:
- **`presence_with_region.f1`** — the core number. The bar the finetuned 3B must beat.
- **`format_valid_rate`** — how often the base model emits parseable JSON (finetuning usually wins here most).
- **`uncertain_with_region` / `progression`** — the nuance the base model tends to miss.

Compare against the finetuned 3B's `sg_eval.json` (from the finetune notebook). If a big model's
zero-shot F1 already matches the finetuned 3B → prompting a bigger model is an option; if it's clearly
lower → finetuning the 3B is the right call.

In [ ]:
import json, glob
for p in sorted(glob.glob("/kaggle/working/sg_eval_zeroshot_*.json")):
    r = json.load(open(p))
    pr = r["presence_with_region"]
    print("%-28s F1=%.3f P=%.3f R=%.3f | format=%.3f | uncertainF1=%.3f"
          % (r["model"].split("/")[-1], pr["f1"], pr["precision"], pr["recall"],
             r["format_valid_rate"], r["uncertain_with_region"]["f1"]))
